# Universidad Central · Big Data · Sesión 13
# ¿Cuándo conviene distribuir tareas y entrenar un modelo?

**Caso principal:** Laura / Compras Claras. **Entorno:** Codespaces con el perfil S13.

[Presentación](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) · [Datos SECOP](../Datos/tc2_corpus_2026.json)

Usaremos **SECOP real** para practicar particiones y tareas. Para enseñar **K-means**, generaremos **puntos numéricos simulados**: así se entiende el algoritmo sin transformar textos ni crear archivos Python adicionales.

**Producto:** comparación pandas–Dask, Parquet SECOP y un modelo K-means **local frente a distribuido**, ejecutados directamente en este cuaderno.

## Objetivo y recorrido

Responder cuatro preguntas: ¿qué es un worker?, ¿cómo se reparte una tarea?, ¿cómo se entrena **un único modelo clásico** entre varios workers?, ¿cuándo la distribución agrega costo en lugar de ahorrar tiempo?

**Ruta:** revisar CPU y memoria → Dask con ocho lotes SECOP → particiones y MapReduce → pandas frente a Dask → K-means numérico, local y distribuido → interpretar tamaños y tiempos.

No usamos modelos neuronales, búsqueda de textos, herramientas externas ni experimentos masivos para esta práctica.

## Cinco estaciones dentro del mismo cuaderno

1. **Codespaces:** CPU lógicas, memoria y configuración de workers.
2. **Distribución:** ocho lotes SECOP con trabajo real y espera simulada, debidamente diferenciados.
3. **MapReduce:** combinar resultados parciales de manera correcta.
4. **Datos:** pandas y Dask sobre las mismas filas; exportar Parquet.
5. **Modelo clásico:** generar puntos numéricos, entrenar K-means local y Dask-ML y decidir cuándo conviene distribuir.

Este cuaderno contiene la ejecución del modelo. No hace falta abrir Streamlit ni copiar scripts adicionales.

## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


**Cómo se lee:** A suma 30; B suma 30. Se encuentran 200 expedientes de referencia y 1.000 procesos etiquetados.  
**Qué nos dice:** pandas y Dask están disponibles y pueden ejecutar un cálculo.  
**Qué NO permite concluir:** que haya un clúster físico o mayor rendimiento que pandas.  
**Error común:** asumir que «pip instaló» equivale a «el flujo completo funciona».


## Laboratorio central · ¿qué trabajo se puede repartir?

**Problema:** ocho lotes independientes de títulos SECOP. Los workers Dask ejecutan realmente las tareas; el modo `texto` cuenta términos reales y el modo `espera` agrega deliberadamente 0,6 segundos de espera **simulada** por tarea. Este último permite estudiar concurrencia, pero **no ejecuta OCR ni transcripción**.

**Cómo interpretar el cronómetro:** compara tareas secuenciales, Dask con workers ya iniciados y Dask incluyendo arranque. Comprueba siempre igualdad del resultado.

**Un clúster local de Dask** es el scheduler más sus workers coordinados en **una máquina virtual**. El nombre clúster no implica varias máquinas físicas. Los workers comparten CPU y RAM del Codespace.

Desde la terminal, si deseas diagnosticar la VM: `nproc`, `free -h`. Todas las ejecuciones de esta clase están en este cuaderno; no es necesario copiar otros scripts.

In [ ]:
import os
import subprocess
import sys
import json
from pathlib import Path

print("CPU lógicas del sistema:", os.cpu_count())
disponibles = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
print("CPU lógicas permitidas por afinidad:", disponibles)
from dask_cuando_si import cpus_disponibles
print("Límite operativo orientativo (afinidad + cuota cgroup):", cpus_disponibles())
print("Para inspeccionar físicamente la VM ejecuta en terminal: nproc; lscpu; free -h")
print("Una VM puede tener 1, 2 o 4 workers, cada uno con uno o varios hilos.")


In [ ]:
# Demo pedagógica pequeña: 8 tareas con Dask Distributed real y espera externa SIMULADA.
# No activa Ollama, 50.000 fichas ni ningún servicio externo.
DEMO_DASK_EN_RUN_ALL = True
WORKERS_DEMO_S13 = 2  # Cambia SOLO este número a 4 para repetir la demostración.
script_dask = RAIZ / "Laboratorios/S13_Codespaces/dask_cuando_si.py"
assert script_dask.exists(), script_dask
if DEMO_DASK_EN_RUN_ALL:
    prueba = subprocess.run(
        [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
         "--tareas", "8", "--espera", "0.6"],
        check=True, cwd=RAIZ, capture_output=True, text=True, timeout=120,
    )
    print(prueba.stdout)
    marca = "RESULTADO_S13_DASK="
    linea = next((l[len(marca):] for l in prueba.stdout.splitlines()
                  if l.startswith(marca)), None)
    assert linea, "No se obtuvo JSON verificable de Dask"
    resumen_demo = json.loads(linea)
    assert resumen_demo["workers"] == WORKERS_DEMO_S13
    assert resumen_demo["resultados_identicos"]
    assert len(resumen_demo["workers_reales"]) == WORKERS_DEMO_S13
    print("Ganador con clúster caliente:", resumen_demo["ganador_caliente"])
    print("Ganador incluyendo arranque:", resumen_demo["ganador_total"])
else:
    print("Para activar esta demostración ligera cambia DEMO_DASK_EN_RUN_ALL=True.")


In [ ]:
# Procesamiento REAL de 1000 títulos SECOP: conteo de vocabulario por lote.
# Se usan los mismos trabajadores; no hay espera simulada, ni OCR, ni modelos.
prueba_texto = subprocess.run(
    [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
     "--tareas", "8", "--modo", "texto"], check=True, cwd=RAIZ,
    capture_output=True, text=True, timeout=120,
)
linea = next((l.split("RESULTADO_S13_DASK=", 1)[1]
              for l in prueba_texto.stdout.splitlines()
              if l.startswith("RESULTADO_S13_DASK=")), None)
assert linea, prueba_texto.stdout[-1800:]
resumen_texto = json.loads(linea)
assert resumen_texto["resultados_identicos"]
assert sum(resumen_texto["terminos_distintos_por_lote"]) > 0
print("Títulos SECOP:", resumen_texto["filas_procesadas"])
print("Workers reales:", len(resumen_texto["workers_reales"]))
print("Segundos secuencial:", resumen_texto["segundos_secuencial"])
print("Segundos Dask caliente:", resumen_texto["segundos_dask_cluster_caliente"])
print("Segundos Dask con arranque:", resumen_texto["segundos_dask_total_incluido_arranque"])
print("Ganador con arranque:", resumen_texto["ganador_total"])


**Cómo se lee:** esta vez Dask hace una transformación auténtica: cuenta los términos de hasta 1.000 títulos SECOP, reparte lotes entre workers y verifica igualdad de diccionarios de frecuencias. No hay pausas artificiales.

**Qué nos dice:** una tarea pequeña puede perder frente a la ejecución local por el costo de iniciar y coordinar workers.

**Qué NO permite concluir:** que Dask no sirva cuando hay datos mayores, operaciones costosas o múltiples máquinas.

**Error común:** confundir ganar en una espera concurrente simulada con ganar en cualquier función de procesamiento.


**Cómo se lee:** el JSON indica número real de workers, CPU habilitadas, filas y palabras procesadas, tiempo secuencial, tiempo de arranque, tiempo Dask caliente y tiempo Dask completo.  
**Qué demuestra:** Dask ejecutó trabajo real en procesos worker y el resultado fue idéntico. La espera fue **simulada**, no es una medición real de OCR.  
**Qué no demuestra:** que Dask sea siempre más rápido en cualquier equipo o con un modelo neuronal.  
**Prueba adicional:** modifica `--workers 2` a `--workers 4` solo si la VM habilita al menos 4 CPU lógicas; observa si cambia la aceleración y el consumo de memoria.

**¿Por qué enseñamos Dask?** Porque puede distribuir funciones Python repetibles, procesar particiones de datos y coordinar tareas independientes. A partir de ahora podrás sustituir la función de demostración por una función real de extracción o clasificación por archivo.


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## 4 · Dataset preparado para demostrar particiones (sin auditoría estudiantil)

En S13 utilizaremos la **muestra SECOP versionada en el repositorio** como insumo para operaciones sobre tablas y textos. El material está preparado para el laboratorio: sus reglas de auditoría de fuente, ceros, identificadores y clasificaciones administrativas son asuntos de validación técnica docente, no actividades evaluables para el estudiante.

En clase importa **cómo Dask procesa una función sobre bloques, cómo se distribuye en varios workers, cómo se combina el resultado y cuánto tiempo cuesta hacerlo**.


In [ ]:
referencia = leer_corpus()
print("Expedientes del taller:", len(referencia))
print(referencia[["id_proceso", "entidad"]].head(3))
assert referencia["id_proceso"].is_unique


### 4A · Usar categorías para contar y agrupar (NO entrenar otro modelo)

El CSV SECOP versionado contiene `descripci_n_del_procedimiento` y `modalidad_de_contratacion`. Aquí **modalidad** se usa exclusivamente para practicar `groupby` y comparar pandas frente a Dask sobre los mismos registros.

**Objetivo observable:** que la suma de las categorías por modalidad coincida exactamente entre ambos motores, independientemente de cómo se distribuyan las particiones.

**No entrenaremos un clasificador por modalidad.** La ampliación opcional del final, si decides activarla, se refiere únicamente a categorías administrativas existentes.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


**Cómo se lee:** 1.000 filas, 59 columnas; «Contratación directa» y «Contratación régimen especial» son clases frecuentes.  
**Qué nos dice:** hay etiquetas registradas y texto para plantear una clasificación didáctica.  
**Qué NO permite concluir:** que esas frecuencias representen todo SECOP ni que el texto contenga evidencia de fraude.  
**Error común:** entrenar sobre los 200 expedientes del Taller 2 como si tuvieran esa etiqueta.


## 5 · pandas primero: la referencia de corrección (conteo de modalidades, NO clasificación)

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## 7 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## 8 · Un clúster para tareas y para modelos

**Clúster de cómputo** = scheduler que coordina tareas + workers que las ejecutan + recursos (CPU, RAM y, cuando corresponde, red). `LocalCluster` crea los workers en la **misma máquina**; en un clúster multinodo los workers pueden vivir en máquinas separadas con memoria propia. Dos workers no equivalen a dos equipos.

**Atención a las palabras:** *clúster Dask* es infraestructura de cómputo; *clúster K-means* es un grupo de observaciones que aprendió el algoritmo.

| Modelo clásico | ¿Se distribuye un solo entrenamiento? | Cuándo considerar Dask |
|---|---|---|
| **K-means Dask-ML** | **Sí**: workers participan en actualizar centroides | Muchas observaciones o cálculos por iteración elevados |
| Regresión lineal/logística Dask-ML | Sí, operaciones distribuidas de la implementación | Matrices grandes y recursos suficientes |
| XGBoost-Dask | **Sí**: un modelo de árboles compartido | Entrenamiento costoso por filas, columnas o árboles |
| Validación cruzada | No: distribuye **varios entrenamientos** | Muchos modelos independientes |
| DBSCAN de scikit-learn | **No**: sigue siendo local | No existe DBSCAN CPU estándar nativo de Dask-ML |

El beneficio depende del tamaño, de los cálculos y del costo de coordinación. **Más workers no fabrican memoria ni CPU**. Los estudiantes ejecutarán solo K-means.

## 9 · Tres cantidades distintas: datos, modelo y memoria de entrenamiento

**Datos:** una matriz de 20.000 filas × 20 variables de ocho bytes ocupa **3,2 MB** (solo los números). **Parámetros de K-means:** cuatro centroides × 20 valores × ocho bytes = **640 bytes** (solo la matriz de centroides). **Entrenamiento:** el proceso ocupa memoria adicional en Python, bibliotecas, asignaciones temporales, copias y, con Dask, scheduler y workers. Esta última cantidad **hay que medirla**, no multiplicar los 640 bytes.

Empezamos con **1.000 puntos y dos variables** para ver los grupos. Puedes ampliar manualmente a 20.000 × 20, pero nunca suponer que eso garantiza una aceleración.

**Datos simulados:** `make_blobs` genera puntos repetibles con semilla fija. No representan procesos SECOP; los datos SECOP reales aparecen en la práctica previa de particiones.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from sklearn.datasets import make_blobs

REGISTROS = 1000       # Extensión voluntaria: 20000
VARIABLES = 2          # Extensión voluntaria: 20
GRUPOS = 4
from dask_cuando_si import cpus_disponibles
CPU_EFECTIVAS = max(1, cpus_disponibles())
WORKERS_MODELO = min(2, CPU_EFECTIVAS)
assert REGISTROS in (1000, 20000) and VARIABLES in (2, 20)
assert 1 <= WORKERS_MODELO <= CPU_EFECTIVAS
X_modelo, _ = make_blobs(n_samples=REGISTROS, n_features=VARIABLES,
                         centers=GRUPOS, cluster_std=1.0, random_state=42)
# El límite de cgroup puede ser menor que la RAM del host.
def limite_memoria_bytes():
    candidatos = [psutil.virtual_memory().total]
    for ruta in ("/sys/fs/cgroup/memory.max",
                 "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            cantidad = int(open(ruta, encoding="utf-8").read().strip())
            if 0 < cantidad < 2**55:
                candidatos.append(cantidad)
        except (OSError, ValueError):
            pass
    return min(candidatos)

MEMORIA_ENTORNO = limite_memoria_bytes()
print(f"Codespace: {CPU_EFECTIVAS} CPU efectivas; RAM límite estimada: {MEMORIA_ENTORNO / 2**30:.2f} GiB")
print(f"Dask: {WORKERS_MODELO} worker(s) en UNA misma máquina")
print(f"Matriz simulada {X_modelo.shape}: {X_modelo.nbytes/1e6:.4f} MB de valores")
print(f"4 centroides teóricos: {GRUPOS * VARIABLES * X_modelo.itemsize} bytes")
display(pd.DataFrame(X_modelo[:, :2], columns=["Variable 1", "Variable 2"]).head())
if VARIABLES == 2:
    plt.figure(figsize=(7, 3.6))
    plt.scatter(X_modelo[:, 0], X_modelo[:, 1], s=8, alpha=0.55)
    plt.title("Puntos simulados antes de agrupar")
    plt.xlabel("Variable 1"); plt.ylabel("Variable 2")
    plt.show()
else:
    print("La representación 2D se omite: hay 20 variables.")

## 10 · Cómo trabaja K-means: asignación → actualización

1. Elegimos cuatro centroides iniciales reproducibles.
2. **Asignar:** cada punto se asocia con el centroide más cercano.
3. **Actualizar:** cada grupo calcula sumas y cantidades; el centroide nuevo es el promedio.
4. **Repetir:** hasta estabilizar los grupos o alcanzar el máximo de iteraciones.

La siguiente celda visualiza **una iteración con NumPy**, sin entrenar otro modelo ni iniciar más workers. Sirve para comprender por qué el entrenamiento puede repartirse. Después compararemos scikit-learn y Dask-ML sobre exactamente los mismos puntos, semilla y centroides iniciales.

In [ ]:
# Ilustración de una iteración: no sustituye al ajuste del modelo.
centros_iniciales = X_modelo[
    np.random.default_rng(42).choice(REGISTROS, GRUPOS, replace=False)
].copy()
if VARIABLES == 2:
    distancias = ((X_modelo[:, None, :] - centros_iniciales[None, :, :])**2).sum(axis=2)
    asignacion_1 = distancias.argmin(axis=1)
    centros_1 = np.vstack([
        X_modelo[asignacion_1 == i].mean(axis=0)
        if (asignacion_1 == i).any() else centros_iniciales[i]
        for i in range(GRUPOS)
    ])
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.7), sharex=True, sharey=True)
    axs[0].scatter(X_modelo[:, 0], X_modelo[:, 1], s=6, c=asignacion_1, alpha=.45)
    axs[0].scatter(centros_iniciales[:, 0], centros_iniciales[:, 1],
                   marker="X", c="black", s=110)
    axs[0].set_title("1. Asignar al centro más cercano")
    axs[1].scatter(X_modelo[:, 0], X_modelo[:, 1], s=6, c=asignacion_1, alpha=.45)
    axs[1].scatter(centros_1[:, 0], centros_1[:, 1], marker="X", c="black", s=110)
    axs[1].set_title("2. Recalcular los centroides")
    for ax in axs: ax.set_xlabel("Variable 1"); ax.set_ylabel("Variable 2")
    plt.tight_layout(); plt.show()
else:
    print("Pasos del algoritmo explicados arriba; no proyectamos 20 variables a 2 sin avisar.")

## 11 · K-means local y distribuido: una comparación justa

**Los datos y parámetros no cambian:** ambos modelos reciben `X_modelo`, cuatro centroides iniciales idénticos, una inicialización y 15 iteraciones como máximo. Las implementaciones internas pueden terminar en soluciones diferentes; por eso también medimos el acuerdo de grupos (ARI).

**Tres celdas en orden:** A) definir el medidor RSS y entrenar local; B) crear el clúster y entrenar Dask-ML; C) comparar los resultados. Ejecutarlas seguidas evita perder el estado del experimento.

**Las fases sí importan:** registrar `fit` y predicción local; para Dask, arranque, transferencia, `fit` y predicción. La comparación del **tiempo total** debe incluir las mismas operaciones de ambos lados; también mostramos el ahorro potencial con el clúster caliente.

**Memoria real:** muestreamos el RSS (memoria residente observada por el sistema operativo) del notebook y de los workers mientras se entrena. El máximo obtenido es un **pico observado por muestreo**, no un máximo absoluto ni la memoria física sumada del clúster: las bibliotecas compartidas pueden contarse en más de un proceso. El modelo de K-means puede terminar ocupando pocos bytes de centroides y haber exigido bastante más durante el ajuste.

In [ ]:
# A. Definir el medidor RSS y entrenar K-means local.
# El RSS del proceso no equivale a la memoria exclusiva del modelo.
import time
import threading
import dask.array as da
from distributed import Client, LocalCluster, wait, get_task_stream
from dask_ml.cluster import KMeans as KMeansDask
from sklearn.cluster import KMeans as KMeansLocal
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits

# No descargar resultados gigantes: cada muestra contiene pocos procesos.
class MuestraRSS:
    def __init__(self, procesos, intervalo=0.05):
        self.procesos, self.intervalo = procesos, intervalo
        self.cierre = threading.Event()
        self.registros = []
    def medir(self):
        fila = {"s": time.perf_counter()}
        for nombre, pid in self.procesos.items():
            try:
                fila[nombre] = psutil.Process(pid).memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                fila[nombre] = 0
        self.registros.append(fila)
    def iniciar(self):
        self.medir()
        self.hilo = threading.Thread(target=self._muestrear, daemon=True)
        self.hilo.start()
    def _muestrear(self):
        while not self.cierre.wait(self.intervalo):
            self.medir()
    def terminar(self):
        self.cierre.set()
        self.hilo.join(timeout=1)
        self.medir()
    def pico_por_proceso(self):
        return {nombre: max(r.get(nombre, 0) for r in self.registros)
                for nombre in self.procesos}

proceso_notebook = os.getpid()
monitor_local = MuestraRSS({"Notebook": proceso_notebook})
monitor_local.iniciar()
inicio = time.perf_counter()
try:
    with threadpool_limits(limits=1):
        modelo_local = KMeansLocal(
            n_clusters=GRUPOS, init=centros_iniciales.copy(),
            n_init=1, max_iter=15, random_state=42
        ).fit(X_modelo)
    tiempo_fit_local = time.perf_counter() - inicio
    t0 = time.perf_counter()
    with threadpool_limits(limits=1):
        etiquetas_locales = modelo_local.predict(X_modelo)
    tiempo_predecir_local = time.perf_counter() - t0
finally:
    monitor_local.terminar()

# LocalCluster = scheduler y workers en el MISMO Codespace.


In [ ]:
# B. Entrenar un único K-means con Dask-ML, dentro de una sola VM.
# Dask tiene un costo de arranque y transferencia que debe registrarse.
inicio_cluster = time.perf_counter()
with LocalCluster(n_workers=WORKERS_MODELO, threads_per_worker=1,
                  processes=True, dashboard_address=None) as cluster:
    with Client(cluster) as client:
        client.wait_for_workers(WORKERS_MODELO)
        workers_info = client.scheduler_info()["workers"]
        direcciones_workers = sorted(workers_info)
        limites_workers = {direccion: int(info["memory_limit"])
                           for direccion, info in workers_info.items()}
        pids_workers = client.run(os.getpid)
        tiempo_arranque = time.perf_counter() - inicio_cluster
        X_distribuido = da.from_array(
            X_modelo, chunks=(max(100, REGISTROS // 8), VARIABLES)
        )
        t0 = time.perf_counter()
        X_distribuido = client.persist(X_distribuido)
        wait(X_distribuido)
        tiempo_transferencia = time.perf_counter() - t0
        monitor_dask = MuestraRSS({
            "Notebook": proceso_notebook,
            **{f"Worker {i+1}": pids_workers[direccion]
               for i, direccion in enumerate(direcciones_workers)}
        })
        monitor_dask.iniciar()
        try:
            modelo_distribuido = KMeansDask(
                n_clusters=GRUPOS, init=centros_iniciales.copy(),
                n_init=1, max_iter=15, random_state=42
            )
            t0 = time.perf_counter()
            with get_task_stream(client=client) as tareas:
                modelo_distribuido.fit(X_distribuido)
            tiempo_fit_dask = time.perf_counter() - t0
            t0 = time.perf_counter()
            etiquetas_dask = np.asarray(
                modelo_distribuido.predict(X_distribuido).compute(), dtype=int
            )
            tiempo_predecir_dask = time.perf_counter() - t0
        finally:
            monitor_dask.terminar()
        tareas_worker = {}
        for tarea in tareas.data:
            direccion = tarea.get("worker")
            if direccion:
                tareas_worker[direccion] = tareas_worker.get(direccion, 0) + 1


In [ ]:
# C. Comparar tiempos de las mismas operaciones y verificar el resultado.
tiempo_total_local = tiempo_fit_local + tiempo_predecir_local
tiempo_total_dask = (tiempo_arranque + tiempo_transferencia +
                     tiempo_fit_dask + tiempo_predecir_dask)
tiempo_dask_caliente = (tiempo_transferencia + tiempo_fit_dask +
                       tiempo_predecir_dask)
acuerdo = adjusted_rand_score(etiquetas_locales, etiquetas_dask)
tiempos = pd.DataFrame([
    {"Método": "Local", "Arranque (s)": 0.,
     "Preparar en workers (s)": 0.,
     "Entrenar (s)": tiempo_fit_local, "Predecir (s)": tiempo_predecir_local,
     "Total comparable (s)": tiempo_total_local},
    {"Método": "Dask", "Arranque (s)": tiempo_arranque,
     "Preparar en workers (s)": tiempo_transferencia,
     "Entrenar (s)": tiempo_fit_dask, "Predecir (s)": tiempo_predecir_dask,
     "Total comparable (s)": tiempo_total_dask},
])
display(tiempos.round(4))
print("Workers reales:", direcciones_workers)
print("Límites RAM por worker (MiB):",
      {k: round(v/2**20, 1) for k, v in limites_workers.items()})
print("Tareas observadas durante fit:", tareas_worker)
print(f"ARI entre asignaciones: {acuerdo:.3f} | Dask caliente: {tiempo_dask_caliente:.3f} s")
print("Ganador total medido:", "Local" if tiempo_total_local < tiempo_total_dask
      else "Dask" if tiempo_total_dask < tiempo_total_local else "Empate")
assert len(etiquetas_locales) == len(etiquetas_dask) == REGISTROS
assert np.isfinite(np.asarray(modelo_distribuido.cluster_centers_)).all()


## 12 · Memoria: qué debemos interpretar

La siguiente celda separa **tamaño lógico de datos**, **centroides aprendidos** y **RSS máximo observado** durante el entrenamiento. No sumes el RSS de workers como si fueran bytes físicos exclusivos: pueden compartir bibliotecas y páginas de memoria; tampoco midas el RSS del notebook como si fuera únicamente K-means.

**RAM por worker:** el límite que informa Dask aplica a ese proceso, no es memoria adicional creada. A partir de ciertos umbrales Dask puede empezar a mover datos gestionados a disco, pausar y reiniciar workers (predeterminados 60/70/80/95 %, según tipo de memoria); no garantiza rescatar todas las asignaciones temporales.

**Cuándo tiene sentido distribuir:** cuando el cálculo por iteración es suficientemente exigente, los datos no caben cómodamente en memoria de un proceso o hay muchos trabajos independientes. Para 1.000×2 o 20.000×20, puede ganar scikit-learn local.

In [ ]:
# Memoria REAL observada por muestreo, no estimación inventada.
resumen_memoria = pd.DataFrame([
    {"Concepto": "Matriz de entrada", "Medida": "Bytes de valores",
     "Valor": X_modelo.nbytes, "Unidad": "bytes"},
    {"Concepto": "Centroides entrenados (local)", "Medida": "Bytes de valores",
     "Valor": modelo_local.cluster_centers_.nbytes, "Unidad": "bytes"},
    {"Concepto": "Centroides entrenados (Dask)", "Medida": "Bytes de valores",
     "Valor": np.asarray(modelo_distribuido.cluster_centers_).nbytes, "Unidad": "bytes"},
])
display(resumen_memoria)
datos_rss = [
    {"Método": "Local", "Proceso": nombre, "RSS pico observado (MiB)": valor/2**20}
    for nombre, valor in monitor_local.pico_por_proceso().items()
] + [
    {"Método": "Dask", "Proceso": nombre, "RSS pico observado (MiB)": valor/2**20}
    for nombre, valor in monitor_dask.pico_por_proceso().items()
]
tabla_rss = pd.DataFrame(datos_rss)
display(tabla_rss.round(2))
plt.figure(figsize=(9, 3.6))
plt.barh(tabla_rss["Método"] + " · " + tabla_rss["Proceso"],
         tabla_rss["RSS pico observado (MiB)"])
plt.xlabel("MiB de RSS, máximo observado durante muestreo")
plt.title("Memoria de procesos observada (no memoria exclusiva del modelo)")
plt.tight_layout(); plt.show()
print("Muestras RSS: local", len(monitor_local.registros),
      "· Dask", len(monitor_dask.registros))
print("Nota: muestreo cada 0,05 s; picos entre muestras pueden no detectarse.")
print("RSS suma bibliotecas compartidas si se agregan procesos; no sumar como RAM física.")

In [ ]:
# Agrupación final; gráfico solo si los datos tienen exactamente dos variables.
if VARIABLES == 2:
    plt.figure(figsize=(7, 3.7))
    plt.scatter(X_modelo[:, 0], X_modelo[:, 1], c=etiquetas_dask,
                s=7, alpha=0.65)
    centros = np.asarray(modelo_distribuido.cluster_centers_)
    plt.scatter(centros[:, 0], centros[:, 1], marker="X", s=130,
                c="black", label="Centroides entrenados")
    plt.title("K-means Dask: agrupación final de puntos simulados")
    plt.xlabel("Variable 1"); plt.ylabel("Variable 2")
    plt.legend(); plt.show()
else:
    print("Se omite la gráfica en dos ejes: el modelo usó 20 variables.")
print("Registros por grupo:", np.bincount(etiquetas_dask.astype(int), minlength=GRUPOS))

## 13 · Escala, límites y decisión

| Matriz numérica `float64` | Memoria de valores | Interpretación |
|---|---:|---|
| 1.000 × 2 | 0,016 MB | Pequeña; local probablemente más sencillo |
| 20.000 × 20 | 3,2 MB | Pequeña por RAM; medir CPU e iteraciones |
| 300.000 × 20 | 48 MB | Todavía puede caber en un proceso local |
| 1.000.000 × 100 | 800 MB | Considerar copias, carga de CPU y RAM efectiva |

Estas cifras no incluyen datos originales, estructuras internas, memoria de procesos ni copias distribuidas. **No existe un umbral universal** de filas que convierta automáticamente un problema en distribuido. Se evalúan CPU, columnas, centroides, iteraciones, comunicación y memoria disponible.

**Un modelo costoso no necesariamente queda grande:** en K-means los centroides son pequeños, mientras el esfuerzo de entrenamiento depende aproximadamente de filas × variables × grupos × iteraciones.

La pregunta final no es «¿funcionó Dask?», sino **«¿tiene sentido pagar la coordinación para este trabajo y estos recursos?»**.

## 14 · Transferencia: convierte la técnica en tu propio proyecto

No memorices rutas del curso. En un proyecto nuevo identifica:

- **Archivo de entrada**: CSV o Parquet que controles y tengas permiso de usar.
- **Clave única**: ID que representa una observación.
- **Columna de agrupación**: qué necesitas comparar y por qué.
- **Etiqueta**: una variable que **exista realmente** si vas a entrenar supervisadamente.
- **Texto o variables**: de dónde provienen; ¿alguna revela directamente la etiqueta?
- **Control simple**: pandas antes de Dask.
- **Salida**: Parquet + manifiesto con fuente, conteos y decisiones.

En `Laboratorios/S13_Codespaces/analisis.py` puedes reemplazar `CSV`, `COL_TEXTO`, `COL_LABEL` y `CLASES` después de inspeccionar el nuevo esquema. Si la columna no existe, debes recibir un error visible, no fabricar valores.


## 15 · Autocomprobación técnica

| Caso | Primera alternativa |
|---|---|
| 200 filas para contar | pandas |
| Muchos archivos independientes | Dask si medir confirma beneficio |
| Un K-means pequeño | scikit-learn |
| Un solo K-means con muchos vectores | Dask-ML KMeans y medir |
| Varias configuraciones independientes | Dask Futures |
| Transformaciones Parquet en clúster Spark | PySpark (S14) |

**Distingue:** repartición de datos, ejecución de modelos independientes y entrenamiento distribuido de **un único modelo**.

In [ ]:
# Reto de cierre · respuesta estructurada y verificable (sin texto libre)
# Cambia SOLO ambos números según tu comparación de los modos texto / espera.
# Esta decisión no requiere métricas de clasificación.
# Los tiempos vienen de ESTE Codespace; su alcance está en el JSON.
ARQUITECTURA = None
LIMITE = None

OPCIONES_ARQUITECTURA = {
    1: "Secuencial cuando las tareas son demasiado pequeñas",
    2: "Dask 2 workers para lotes independientes con espera",
    3: "Comparar 4 workers y arranque antes de decidir",
}
OPCIONES_LIMITE = {
    1: "La aceleración caliente excluye arranque y preparación",
    2: "La espera OCR/ASR fue simulada, no un servicio real",
    3: "Workers de una VM comparten CPU y memoria",
    4: "Los tiempos dependen de los recursos del Codespace usado",
}
print("Elige UNA arquitectura y UN límite de los siguientes menús:")
print("Arquitectura:", OPCIONES_ARQUITECTURA)
print("Límite:", OPCIONES_LIMITE)
if ARQUITECTURA is None or LIMITE is None:
    print("Práctica pendiente: cambia ambos None por los índices seleccionados y vuelve a ejecutar.")
else:
    if type(ARQUITECTURA) is not int or ARQUITECTURA not in OPCIONES_ARQUITECTURA:
        raise ValueError("ARQUITECTURA debe ser exactamente 1, 2 o 3")
    if type(LIMITE) is not int or LIMITE not in OPCIONES_LIMITE:
        raise ValueError("LIMITE debe ser exactamente 1, 2, 3 o 4")
    # La demostración de espera puede desactivarse: no referenciar variables inexistentes.
    # Tampoco se permite exportar una decisión basada en resultados no ejecutados.
    medidas_s13 = {
        "texto": globals().get("resumen_texto"),
        "espera": globals().get("resumen_demo"),
    }
    if not all(
        isinstance(resultado, dict) and resultado.get("resultados_identicos") is True
        for resultado in medidas_s13.values()
    ):
        raise RuntimeError(
            "Falta evidencia real para uno de los dos modos. "
            "Ejecuta la celda de DEMO_DASK_EN_RUN_ALL=True y la celda "
            "de procesamiento de texto real, y vuelve a esta decisión."
        )
    import json
    from pathlib import Path
    evidencia = {
        "sesion": 13,
        "arquitectura_codigo": ARQUITECTURA,
        "arquitectura": OPCIONES_ARQUITECTURA[ARQUITECTURA],
        "limite_codigo": LIMITE,
        "limite": OPCIONES_LIMITE[LIMITE],
        "fuente": "mediciones reales de esta ejecución en Codespaces, con cronómetros separados",
        "modos_medidos": medidas_s13
    }
    salida_decision = RAIZ / "Laboratorios/S13_Codespaces/salidas/decision_s13.json"
    salida_decision.parent.mkdir(parents=True, exist_ok=True)
    salida_decision.write_text(json.dumps(evidencia,ensure_ascii=False,indent=2), encoding="utf-8")
    print("Evidencia exportada a:", salida_decision)
    print("Comprueba que la arquitectura seleccionada corresponda a tus tiempos, no a un resultado supuesto.")


## Cierre · Puente a S14

Resultados de S13: Parquet generado a partir de **SECOP real**, evidencia de Dask y un **K-means de puntos simulados** para comprender un único entrenamiento distribuido. No se mezclan los orígenes.

Aprendimos a distinguir worker de máquina física, trabajo distribuido de aceleración, y un único modelo distribuido de muchos entrenamientos independientes. S14 introduce PySpark sobre Parquet y permite comparar su K-means nativo.

Fuentes: [Dask](https://docs.dask.org/) · [Dask-ML KMeans](https://ml.dask.org/modules/generated/dask_ml.cluster.KMeans.html) · [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

**Memoria evaluada:** tamaño de matriz, tamaño de centroides, pico RSS observado de procesos y límites por worker. Recuerda que el RSS muestreado no equivale al tamaño final del modelo ni a memoria física exclusiva.